# S3 Data Puller

This notebook demonstrates different ways to pull data from Amazon S3.

## Setup and Installation

In [24]:
import boto3
import pandas as pd
import os
from io import StringIO, BytesIO
from dotenv import load_dotenv
import json

In [25]:
# AWS Configuration

load_dotenv()
AWS_ACCESS_KEY_ID = os.getenv('AWS_ACCESS_KEY_ID')  # or set directly
AWS_SECRET_ACCESS_KEY = os.getenv('AWS_SECRET_ACCESS_KEY')  # or set directly
AWS_REGION = os.getenv("AWS_REGION") # Change to your region
AWS_ENDPOINT = os.getenv("AWS_ENDPOINT") # Change to your endpoint

# S3 Configuration
# BUCKET_NAME = 'your-bucket-name'  # Replace with your bucket name
# FILE_KEY = 'path/to/your/file.csv'  # Replace with your file path

## Method 1: Using boto3 Client

In [26]:
# Initialize S3 client
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION,
    endpoint_url=AWS_ENDPOINT
)

print("S3 client initialized successfully")

S3 client initialized successfully


In [27]:
# List objects in bucket
def list_s3_objects(bucket_name, prefix=''):
    """List objects in S3 bucket with optional prefix"""
    try:
        response = s3_client.list_objects_v2(
            Bucket=bucket_name,
            Prefix=prefix,
        )
        
        if 'Contents' in response:
            objects = [obj['Key'] for obj in response['Contents']]
            print(f"Found {len(objects)} objects:")
            for obj in objects[:10]:  # Show first 10
                print(f"  {obj}")
            if len(objects) > 10:
                print(f"  ... and {len(objects) - 10} more")
            return objects
        else:
            print("No objects found")
            return []
    except Exception as e:
        print(f"Error listing objects: {e}")
        return []

# List objects (uncomment to use)
# objects = list_s3_objects(BUCKET_NAME)

In [28]:
# Download file directly to memory
def download_s3_file_to_memory(bucket_name, file_key):
    """Download S3 file to memory and return as bytes"""
    try:
        response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
        return response['Body'].read()
    except Exception as e:
        print(f"Error downloading file: {e}")
        return None

# Example: Download and read CSV
def read_csv_from_s3(bucket_name, file_key):
    """Read CSV file directly from S3 into pandas DataFrame"""
    try:
        csv_obj = s3_client.get_object(Bucket=bucket_name, Key=file_key)
        body = csv_obj['Body']
        csv_string = body.read().decode('utf-8')
        df = pd.read_csv(StringIO(csv_string))
        return df
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return None

# Example usage (uncomment to use)
# df = read_csv_from_s3(BUCKET_NAME, FILE_KEY)
# if df is not None:
#     print(f"DataFrame shape: {df.shape}")
#     print(df.head())

In [ ]:
import re
from tqdm import tqdm

def download_directory(s3, bucket, prefix, target_dir, exclude_pattern = None):
    """Download all objects in S3 bucket with given prefix to local directory"""
    exclude_pattern = re.compile(exclude_pattern) if exclude_pattern else None
    for obj in tqdm(list_s3_objects(bucket, prefix)):
        if obj.endswith('/'):
            continue
        if exclude_pattern and exclude_pattern.search(obj):
            continue
        
        target_path = os.path.join(target_dir, os.path.relpath(obj, prefix))
        if os.path.exists(target_path): continue # skip if exists in local
        os.makedirs(os.path.dirname(target_path), exist_ok=True)
        s3.download_file(bucket, obj, target_path)
        print(f"retrieved {obj} to {target_path}" )

In [30]:
download_directory(s3_client, "runpod", "the-power-of-noise/", "./data", exclude_pattern=r'.*\.npy|.*\.faiss')

Found 204 objects:
  the-power-of-noise/.ipynb_checkpoints/10k_train_dataset-checkpoint.json
  the-power-of-noise/10k_other_random_results_at60.pkl
  the-power-of-noise/10k_random_results_at60.pkl
  the-power-of-noise/10k_train_dataset.json
  the-power-of-noise/adore_search_results_at200.pkl
  the-power-of-noise/bm25_test_search_results_at250.pkl
  the-power-of-noise/contriever_search_results_at150.pkl
  the-power-of-noise/contriever_test_search_results_at150.pkl
  the-power-of-noise/corpus/embeddings/wiki_dec_2018/contriever_10111999_embeddings.npy
  the-power-of-noise/corpus/embeddings/wiki_dec_2018/contriever_10239999_embeddings.npy
  ... and 194 more
testing the-power-of-noise/.ipynb_checkpoints/10k_train_dataset-checkpoint.json
testing the-power-of-noise/10k_other_random_results_at60.pkl
testing the-power-of-noise/10k_random_results_at60.pkl
testing the-power-of-noise/10k_train_dataset.json
testing the-power-of-noise/adore_search_results_at200.pkl
testing the-power-of-noise/bm25_t